# 🎯 Model Stacking & Ensembling

Combinar múltiplos modelos para melhorar robustez e performance.

## 📋 Conteúdo
1. **Stacked Generalization** - Meta-learner combinando XGBoost, LightGBM, CatBoost
2. **Weighted Averaging** - Pesos baseados em performance por região
3. **Blending** - Combinação de previsões
4. **Comparação de Performance**

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import warnings

from sklearn.linear_model import Ridge
from sklearn.ensemble import StackingRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

from src.features.feature_engineering import FeatureEngineer
from src.models.evaluation import ModelEvaluator

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print('✓ Setup completo')

✓ Setup completo


## 1. Carregar Dados e Modelos

In [2]:
# Carregar dados
df = pd.read_parquet('../data/processed/processed_data.parquet')
fe = FeatureEngineer()
df_features = fe.create_all_features(df)

# Preparar features
exclude_cols = ['timestamp', 'consumption_mw', 'region', 'holiday_name']
for col in df_features.columns:
    if pd.api.types.is_datetime64_any_dtype(df_features[col]) or df_features[col].dtype == 'object':
        if col not in exclude_cols:
            exclude_cols.append(col)

features = [c for c in df_features.columns if c not in exclude_cols]

# Split temporal
df_sorted = df_features.sort_values('timestamp').reset_index(drop=True)
n = len(df_sorted)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

X_train = df_sorted.iloc[:train_end][features].values
y_train = df_sorted.iloc[:train_end]['consumption_mw'].values
X_val = df_sorted.iloc[train_end:val_end][features].values
y_val = df_sorted.iloc[train_end:val_end]['consumption_mw'].values
X_test = df_sorted.iloc[val_end:][features].values
y_test = df_sorted.iloc[val_end:]['consumption_mw'].values

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

Creating features...
  - Temporal features
  - Lag features
  - Rolling features
  - Interaction features
  - Removed 240 rows with NaN
Train: 122,475 | Val: 26,245 | Test: 26,245


## 2. Treinar Base Models

In [3]:
print('Treinando base models...')

# XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=0)
xgb_model.fit(X_train, y_train)
print('✓ XGBoost')

# LightGBM
lgb_model = lgb.LGBMRegressor(n_estimators=300, max_depth=8, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=-1)
lgb_model.fit(X_train, y_train)
print('✓ LightGBM')

# CatBoost
cat_model = CatBoostRegressor(iterations=300, depth=8, learning_rate=0.05, random_state=42, verbose=False)
cat_model.fit(X_train, y_train)
print('✓ CatBoost')

base_models = {'XGBoost': xgb_model, 'LightGBM': lgb_model, 'CatBoost': cat_model}

Treinando base models...
✓ XGBoost
✓ LightGBM
✓ CatBoost


## 3. Stacking Regressor

In [4]:
print('Treinando Stacking Regressor...')

estimators = [
    ('xgb', xgb_model),
    ('lgb', lgb_model),
    ('cat', cat_model)
]

stack_model = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

stack_model.fit(X_train, y_train)
print('✓ Stacking completo')

Treinando Stacking Regressor...
✓ Stacking completo


## 4. Weighted Average

In [5]:
# Calcular pesos baseado em RMSE no validation set
evaluator = ModelEvaluator()
weights = {}

for name, model in base_models.items():
    y_val_pred = model.predict(X_val)
    metrics = evaluator.calculate_metrics(y_val, y_val_pred)
    rmse = metrics['rmse']
    weights[name] = 1 / (rmse + 1e-6)  # Inverso do RMSE

# Normalizar pesos
total = sum(weights.values())
weights = {k: v/total for k, v in weights.items()}

print('\nPesos calculados (baseado em RMSE):')
for name, w in weights.items():
    print(f'  {name}: {w:.4f}')


Pesos calculados (baseado em RMSE):
  XGBoost: 0.4358
  LightGBM: 0.3241
  CatBoost: 0.2401


## 5. Avaliar Ensemble Methods

In [6]:
# Previsões individuais no test set
preds = {}
for name, model in base_models.items():
    preds[name] = model.predict(X_test)

# Simple Average
preds['Simple_Avg'] = np.mean([preds['XGBoost'], preds['LightGBM'], preds['CatBoost']], axis=0)

# Weighted Average
preds['Weighted_Avg'] = (weights['XGBoost'] * preds['XGBoost'] + 
                          weights['LightGBM'] * preds['LightGBM'] + 
                          weights['CatBoost'] * preds['CatBoost'])

# Stacking
preds['Stacking'] = stack_model.predict(X_test)

# Avaliar todos
results = []
for name, y_pred in preds.items():
    metrics = evaluator.calculate_metrics(y_test, y_pred)
    results.append({
        'Model': name,
        'MAE': metrics['mae'],
        'RMSE': metrics['rmse'],
        'MAPE': metrics['mape'],
        'R2': metrics['r2']
    })

df_results = pd.DataFrame(results).sort_values('RMSE')

print('\n' + '='*80)
print('COMPARAÇÃO DE MÉTODOS DE ENSEMBLE')
print('='*80)
print(df_results.to_string(index=False))
print('='*80)

best = df_results.iloc[0]
print(f"\n🏆 Melhor método: {best['Model']}")
print(f"   MAPE: {best['MAPE']:.2f}% | R²: {best['R2']:.4f}")


COMPARAÇÃO DE MÉTODOS DE ENSEMBLE
       Model       MAE      RMSE     MAPE       R2
    Stacking 13.349738 23.318400 1.106868 0.999275
     XGBoost 13.311561 23.514349 1.111032 0.999263
Weighted_Avg 16.806957 27.863515 1.458117 0.998965
  Simple_Avg 18.307534 29.593779 1.596129 0.998833
    LightGBM 18.874983 31.004695 1.645872 0.998719
    CatBoost 28.475202 41.833044 2.505850 0.997668

🏆 Melhor método: Stacking
   MAPE: 1.11% | R²: 0.9993


## 6. Salvar Melhor Ensemble

In [7]:
output_dir = Path('../data/models')
output_dir.mkdir(parents=True, exist_ok=True)

# Salvar stacking model
joblib.dump(stack_model, output_dir / 'ensemble_stacking.pkl')
print(f"✓ Modelo salvo: ensemble_stacking.pkl")

# Salvar pesos
import json
with open(output_dir / 'ensemble_weights.json', 'w') as f:
    json.dump(weights, f, indent=2)
print(f"✓ Pesos salvos: ensemble_weights.json")

print('\n✅ Ensemble completo!')

✓ Modelo salvo: ensemble_stacking.pkl
✓ Pesos salvos: ensemble_weights.json

✅ Ensemble completo!
